# 04 — EDA Financeira Integrada: Benner + DeepLegal

Este notebook aprofunda a análise financeira da carteira jurídica cruzando:

1. **Benner**: valor ajuizado interno, carteira, produto, ação, assunto, fase, estimativa de perda, passível de acordo e processo relevante.
2. **DeepLegal Processos**: valor da causa externo, condenação, acordo, resultado, sentença, assunto, classe, vara, órgão julgador, UF/cidade e sinais jurídicos.
3. **DeepLegal Eventos**: trajetória processual agregada, eventos de sentença, recurso, execução, acordo, pagamento, trânsito em julgado etc.

## Pergunta central

> Quais causas/processos fazem mais sentido atacar para reduzir perdas financeiras do banco?

Para responder, vamos combinar:

- valor ajuizado Benner;
- valor DeepLegal;
- divergência entre valores;
- valor de condenação;
- valor de acordo;
- estimativa de perda Benner;
- sinais jurídicos DeepLegal;
- eventos processuais;
- concentração financeira;
- risco financeiro proxy;
- potencial de acordo;
- priorização por carteira, produto, ação, assunto, fase, comarca, UF, escritório e advogado interno.

## Arquivo de entrada recomendado

```text
data/processed/abt_integrada_benner_deeplegal_eda.parquet
```

## 1. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

BASE_DIR = Path("..")  # altere para Path('.') se o notebook estiver na raiz
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INTEGRATED_FILE = PROCESSED_DIR / "abt_integrada_benner_deeplegal_eda.parquet"

# Benner
BENNER_ID_COL = "Identificador"
BENNER_NUM_PROCESSO_COL = "Numerodistribuicao"
BENNER_VALOR_ORIGINAL_COL = "Valorajuizado"
BENNER_VALOR_COL = "valor_ajuizado"
BENNER_CARTEIRA_COL = "Carteira"
BENNER_PRODUTO_COL = "Nm_Produto"
BENNER_ACAO_COL = "Nm_Acao"
BENNER_ASSUNTO_COL = "Nm_Assunto"
BENNER_TIPO_PROCESSO_COL = "Tipoprocesso"
BENNER_UF_COL = "Uf"
BENNER_COMARCA_COL = "Comarca"
BENNER_FASE_COL = "Fasedoprocesso"
BENNER_ESTIMATIVA_PERDA_COL = "Estimativa_De_Perda"
BENNER_PASSIVEL_ACORDO_COL = "Passiveldeacordo"
BENNER_RELEVANTE_COL = "Processorelevante"
BENNER_CREDENCIADO_COL = "Credenciado"
BENNER_ADV_INTERNO_COL = "Adv_Interno"
BENNER_LOCAL_SERVICO_COL = "Localdeservico"

# DeepLegal com prefixo esperado na ABT integrada
DL_VALOR_COL = "dl_valor_valor"
DL_CONDENACAO_VALOR_COL = "dl_condenacao_valor"
DL_VALOR_ACORDO_COL = "dl_valor_do_acordo_valor"
DL_ASSUNTO_COL = "dl_assunto_normalizado_texto"
DL_CLASSE_COL = "dl_classe_texto"
DL_UF_COL = "dl_uf"
DL_CIDADE_COL = "dl_cidade"
DL_VARA_COL = "dl_vara_texto"
DL_ORGAO_COL = "dl_orgao_julgador_texto"
DL_STATUS_COL = "dl_status_texto"
DL_FASE_COL = "dl_fase_processual_texto"
DL_RESULTADO_COL = "dl_resultado_final_processo_text"
DL_RESULTADO_JULGAMENTO_COL = "dl_resultadojulgamento_tipo"
DL_SENTENCA_COL = "dl_sentenca_tipo"

EVENT_SIGNAL_COLS = [
    "dl_flag_teve_sentenca",
    "dl_flag_teve_recurso",
    "dl_flag_teve_execucao",
    "dl_flag_teve_acordo",
    "dl_flag_teve_pagamento",
    "dl_flag_teve_transito_julgado",
    "dl_flag_teve_audiencia",
    "dl_flag_teve_citacao",
    "dl_flag_teve_arquivamento",
]

## 2. Funções utilitárias

In [ ]:
NULL_STRINGS = {"", "null", "nan", "none", "na", "n/a", "-", "não informado", "nao informado", "sem informação", "sem informacao"}

def read_data(path, columns=None, nrows=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path, columns=columns)
    if suffix == ".csv":
        return pd.read_csv(path, usecols=columns, nrows=nrows)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, usecols=columns, nrows=nrows)
    raise ValueError(f"Formato não suportado: {path}")

def normalize_text(text):
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if text in NULL_STRINGS:
        return None
    return text

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def save_report(df, filename):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"Salvo: {path}")

def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def parse_yes_no(value):
    t = normalize_text(value)
    if t in ["sim", "s", "yes", "y", "true", "1"]:
        return 1
    if t in ["nao", "n", "no", "false", "0"]:
        return 0
    return np.nan

def map_loss_estimate(value):
    t = normalize_text(value)
    if t is None:
        return np.nan
    if "remoto" in t:
        return 0
    if "possivel" in t:
        return 1
    if "provavel" in t:
        return 2
    return np.nan

def loss_estimate_weight(value):
    # Pesos exploratórios, não oficiais.
    t = normalize_text(value)
    if t is None:
        return 0.30
    if "remoto" in t:
        return 0.10
    if "possivel" in t:
        return 0.50
    if "provavel" in t:
        return 0.90
    return 0.30

def classify_outcome_text(text):
    t = normalize_text(text)
    if t is None:
        return np.nan
    favorable = [
        "improcedente", "improcedencia", "extinto", "extincao",
        "desistencia", "sem condenacao", "nao procedente"
    ]
    unfavorable = [
        "parcialmente procedente", "parc procedente", "procedente",
        "condenacao", "condenado", "acordo", "pagamento",
        "dano moral", "valor arbitrado"
    ]
    if any(p in t for p in favorable):
        return 0
    if any(p in t for p in unfavorable):
        return 1
    return np.nan

## 3. Carregar ABT integrada

In [ ]:
df = read_data(INTEGRATED_FILE)

print("ABT integrada:", df.shape)
df.head()

## 4. Garantir campos financeiros e alvos auxiliares

In [ ]:
df_fin = df.copy()

# Benner value
if BENNER_VALOR_COL not in df_fin.columns and BENNER_VALOR_ORIGINAL_COL in df_fin.columns:
    df_fin[BENNER_VALOR_COL] = pd.to_numeric(df_fin[BENNER_VALOR_ORIGINAL_COL], errors="coerce")

df_fin["valor_benner"] = pd.to_numeric(df_fin[BENNER_VALOR_COL], errors="coerce")

# DeepLegal values
df_fin["valor_deeplegal"] = pd.to_numeric(df_fin[DL_VALOR_COL], errors="coerce") if DL_VALOR_COL in df_fin.columns else np.nan
df_fin["valor_condenacao_deeplegal"] = pd.to_numeric(df_fin[DL_CONDENACAO_VALOR_COL], errors="coerce") if DL_CONDENACAO_VALOR_COL in df_fin.columns else np.nan
df_fin["valor_acordo_deeplegal"] = pd.to_numeric(df_fin[DL_VALOR_ACORDO_COL], errors="coerce") if DL_VALOR_ACORDO_COL in df_fin.columns else np.nan

# Estimativa de perda Benner
if BENNER_ESTIMATIVA_PERDA_COL in df_fin.columns:
    df_fin["estimativa_perda_score"] = df_fin[BENNER_ESTIMATIVA_PERDA_COL].apply(map_loss_estimate)
    df_fin["flag_perda_provavel"] = (df_fin["estimativa_perda_score"] == 2).astype(float)
    df_fin.loc[df_fin["estimativa_perda_score"].isna(), "flag_perda_provavel"] = np.nan
    df_fin["flag_perda_possivel_ou_provavel"] = (df_fin["estimativa_perda_score"] >= 1).astype(float)
    df_fin.loc[df_fin["estimativa_perda_score"].isna(), "flag_perda_possivel_ou_provavel"] = np.nan
    df_fin["peso_estimativa_perda"] = df_fin[BENNER_ESTIMATIVA_PERDA_COL].apply(loss_estimate_weight)
else:
    df_fin["estimativa_perda_score"] = np.nan
    df_fin["flag_perda_provavel"] = np.nan
    df_fin["flag_perda_possivel_ou_provavel"] = np.nan
    df_fin["peso_estimativa_perda"] = 0.30

# Flags Benner
df_fin["flag_passivel_acordo_benner"] = df_fin[BENNER_PASSIVEL_ACORDO_COL].apply(parse_yes_no).fillna(0) if BENNER_PASSIVEL_ACORDO_COL in df_fin.columns else 0
df_fin["flag_processo_relevante_benner"] = df_fin[BENNER_RELEVANTE_COL].apply(parse_yes_no).fillna(0) if BENNER_RELEVANTE_COL in df_fin.columns else 0

# DeepLegal textual target, caso não exista
if "dl_target_perda_textual" not in df_fin.columns:
    df_fin["dl_target_perda_textual"] = np.nan
    for col in existing_cols(df_fin, [DL_RESULTADO_COL, DL_RESULTADO_JULGAMENTO_COL, DL_SENTENCA_COL]):
        temp = df_fin[col].apply(classify_outcome_text)
        mask = df_fin["dl_target_perda_textual"].isna() & temp.notna()
        df_fin.loc[mask, "dl_target_perda_textual"] = temp.loc[mask]

# Match flags
if "flag_match_deeplegal_processos" not in df_fin.columns:
    df_fin["flag_match_deeplegal_processos"] = df_fin["valor_deeplegal"].notna().astype(int)

if "flag_match_deeplegal_eventos" not in df_fin.columns:
    if "dl_qtd_eventos" in df_fin.columns:
        df_fin["flag_match_deeplegal_eventos"] = df_fin["dl_qtd_eventos"].notna().astype(int)
    else:
        df_fin["flag_match_deeplegal_eventos"] = 0

df_fin[[
    "valor_benner", "valor_deeplegal", "valor_condenacao_deeplegal", "valor_acordo_deeplegal",
    "estimativa_perda_score", "flag_perda_provavel", "flag_perda_possivel_ou_provavel",
    "peso_estimativa_perda", "flag_passivel_acordo_benner", "flag_processo_relevante_benner",
    "dl_target_perda_textual"
]].head()

## 5. Perfil das colunas financeiras

In [ ]:
financial_cols = [
    "valor_benner",
    "valor_deeplegal",
    "valor_condenacao_deeplegal",
    "valor_acordo_deeplegal",
]

rows = []
for col in financial_cols:
    s = pd.to_numeric(df_fin[col], errors="coerce")
    rows.append({
        "coluna": col,
        "qtd_nao_nulos": int(s.notna().sum()),
        "perc_nao_nulos": float(s.notna().mean()),
        "qtd_zeros": int((s == 0).sum()),
        "perc_zeros": float((s == 0).mean()),
        "soma_total": float(s.sum()),
        "media": float(s.mean()),
        "mediana": float(s.median()),
        "p75": float(s.quantile(0.75)),
        "p90": float(s.quantile(0.90)),
        "p95": float(s.quantile(0.95)),
        "p99": float(s.quantile(0.99)),
        "max": float(s.max()),
    })

financial_profile = pd.DataFrame(rows).sort_values("soma_total", ascending=False)
save_report(financial_profile, "financial_profile_integrado_benner_deeplegal.csv")
financial_profile

## 6. Distribuição e concentração do valor

In [ ]:
def pareto_value_analysis(df, value_col):
    temp = df[df[value_col].notna() & (df[value_col] > 0)].copy()
    temp = temp.sort_values(value_col, ascending=False)
    total = temp[value_col].sum()

    if len(temp) == 0 or total == 0:
        return pd.DataFrame()

    temp["rank_percentual"] = np.arange(1, len(temp) + 1) / len(temp)

    rows = []
    for pct in [0.01, 0.05, 0.10, 0.20, 0.30]:
        mask = temp["rank_percentual"] <= pct
        rows.append({
            "coluna_valor": value_col,
            "top_percentual_processos": pct,
            "qtd_processos": int(mask.sum()),
            "valor_total_concentrado": float(temp.loc[mask, value_col].sum()),
            "perc_valor_total_concentrado": float(temp.loc[mask, value_col].sum() / total),
        })

    return pd.DataFrame(rows)

pareto_integrado = pd.concat([
    pareto_value_analysis(df_fin, "valor_benner"),
    pareto_value_analysis(df_fin, "valor_deeplegal"),
    pareto_value_analysis(df_fin, "valor_condenacao_deeplegal"),
    pareto_value_analysis(df_fin, "valor_acordo_deeplegal"),
], ignore_index=True)

save_report(pareto_integrado, "pareto_financeiro_integrado_benner_deeplegal.csv")
pareto_integrado

## 7. Faixas de valor integradas

In [ ]:
def create_value_band_series(s):
    bins = [-np.inf,0,1000,5000,10000,25000,50000,100000,250000,500000,1000000,np.inf]
    labels = [
        "00_sem_valor_ou_zero",
        "01_ate_1k",
        "02_1k_5k",
        "03_5k_10k",
        "04_10k_25k",
        "05_25k_50k",
        "06_50k_100k",
        "07_100k_250k",
        "08_250k_500k",
        "09_500k_1mm",
        "10_acima_1mm"
    ]
    return pd.cut(s.fillna(0), bins=bins, labels=labels, include_lowest=True).astype(str)

df_fin["faixa_valor_benner"] = create_value_band_series(df_fin["valor_benner"])
df_fin["faixa_valor_deeplegal"] = create_value_band_series(df_fin["valor_deeplegal"])
df_fin["faixa_valor_condenacao_deeplegal"] = create_value_band_series(df_fin["valor_condenacao_deeplegal"])
df_fin["faixa_valor_acordo_deeplegal"] = create_value_band_series(df_fin["valor_acordo_deeplegal"])

def value_band_report(df, band_col, value_col):
    out = (
        df.groupby(band_col, dropna=False)
        .agg(
            qtd_processos=(df.columns[0], "count"),
            valor_total=(value_col, "sum"),
            valor_medio=(value_col, "mean"),
            valor_mediano=(value_col, "median"),
            taxa_perda_possivel_ou_provavel=("flag_perda_possivel_ou_provavel", "mean"),
            taxa_perda_provavel=("flag_perda_provavel", "mean"),
            taxa_match_dl_processos=("flag_match_deeplegal_processos", "mean"),
            taxa_match_dl_eventos=("flag_match_deeplegal_eventos", "mean"),
        )
        .reset_index()
    )
    out["perc_processos"] = out["qtd_processos"] / out["qtd_processos"].sum()
    out["perc_valor_total"] = out["valor_total"] / out["valor_total"].sum()
    out["exposicao_risco_proxy"] = out["valor_total"] * out["taxa_perda_possivel_ou_provavel"].fillna(0)
    return out

band_benner = value_band_report(df_fin, "faixa_valor_benner", "valor_benner")
band_deeplegal = value_band_report(df_fin, "faixa_valor_deeplegal", "valor_deeplegal")
band_condenacao = value_band_report(df_fin, "faixa_valor_condenacao_deeplegal", "valor_condenacao_deeplegal")
band_acordo = value_band_report(df_fin, "faixa_valor_acordo_deeplegal", "valor_acordo_deeplegal")

save_report(band_benner, "risk_by_faixa_valor_benner.csv")
save_report(band_deeplegal, "risk_by_faixa_valor_deeplegal.csv")
save_report(band_condenacao, "risk_by_faixa_valor_condenacao_deeplegal.csv")
save_report(band_acordo, "risk_by_faixa_valor_acordo_deeplegal.csv")

band_benner

## 8. Divergência entre valor Benner e valor DeepLegal

In [ ]:
df_fin["diff_valor_benner_deeplegal"] = df_fin["valor_benner"] - df_fin["valor_deeplegal"]
df_fin["abs_diff_valor_benner_deeplegal"] = df_fin["diff_valor_benner_deeplegal"].abs()
df_fin["ratio_valor_benner_deeplegal"] = safe_divide(df_fin["valor_benner"], df_fin["valor_deeplegal"])

df_fin["flag_valor_benner_sem_deeplegal"] = (df_fin["valor_benner"].notna() & df_fin["valor_deeplegal"].isna()).astype(int)
df_fin["flag_valor_deeplegal_sem_benner"] = (df_fin["valor_deeplegal"].notna() & df_fin["valor_benner"].isna()).astype(int)

df_fin["diff_relativa_valor"] = safe_divide(
    df_fin["abs_diff_valor_benner_deeplegal"],
    df_fin[["valor_benner", "valor_deeplegal"]].max(axis=1)
)

df_fin["flag_divergencia_material_valor"] = (
    (df_fin["abs_diff_valor_benner_deeplegal"] >= 10000) &
    (df_fin["diff_relativa_valor"] >= 0.50)
).astype(int)

value_diff_summary = pd.DataFrame([{
    "qtd_com_ambos_valores": int((df_fin["valor_benner"].notna() & df_fin["valor_deeplegal"].notna()).sum()),
    "correlacao_valor_benner_deeplegal": float(df_fin[["valor_benner", "valor_deeplegal"]].corr().iloc[0, 1]),
    "media_abs_diff": float(df_fin["abs_diff_valor_benner_deeplegal"].mean()),
    "mediana_abs_diff": float(df_fin["abs_diff_valor_benner_deeplegal"].median()),
    "p90_abs_diff": float(df_fin["abs_diff_valor_benner_deeplegal"].quantile(0.90)),
    "p95_abs_diff": float(df_fin["abs_diff_valor_benner_deeplegal"].quantile(0.95)),
    "p99_abs_diff": float(df_fin["abs_diff_valor_benner_deeplegal"].quantile(0.99)),
    "qtd_divergencia_material": int(df_fin["flag_divergencia_material_valor"].sum()),
    "perc_divergencia_material": float(df_fin["flag_divergencia_material_valor"].mean()),
}])

save_report(value_diff_summary, "value_divergence_summary_benner_deeplegal.csv")
value_diff_summary

In [ ]:
top_divergencias = (
    df_fin.sort_values("abs_diff_valor_benner_deeplegal", ascending=False)
    [existing_cols(df_fin, [
        BENNER_ID_COL,
        BENNER_NUM_PROCESSO_COL,
        BENNER_CARTEIRA_COL,
        BENNER_PRODUTO_COL,
        BENNER_ACAO_COL,
        BENNER_ASSUNTO_COL,
        BENNER_ESTIMATIVA_PERDA_COL,
        "valor_benner",
        "valor_deeplegal",
        "diff_valor_benner_deeplegal",
        "abs_diff_valor_benner_deeplegal",
        "diff_relativa_valor",
        "ratio_valor_benner_deeplegal",
        DL_ASSUNTO_COL,
        DL_CLASSE_COL,
        DL_FASE_COL,
        DL_STATUS_COL,
    ])]
    .head(300)
)
save_report(top_divergencias, "top_300_divergencias_valor_benner_deeplegal.csv")
top_divergencias.head(50)

## 9. Condenação e acordo DeepLegal

In [ ]:
df_fin["ratio_condenacao_sobre_valor_benner"] = safe_divide(df_fin["valor_condenacao_deeplegal"], df_fin["valor_benner"])
df_fin["ratio_acordo_sobre_valor_benner"] = safe_divide(df_fin["valor_acordo_deeplegal"], df_fin["valor_benner"])
df_fin["ratio_condenacao_sobre_valor_deeplegal"] = safe_divide(df_fin["valor_condenacao_deeplegal"], df_fin["valor_deeplegal"])
df_fin["ratio_acordo_sobre_valor_deeplegal"] = safe_divide(df_fin["valor_acordo_deeplegal"], df_fin["valor_deeplegal"])

df_fin["flag_tem_condenacao_deeplegal"] = (df_fin["valor_condenacao_deeplegal"].fillna(0) > 0).astype(int)
df_fin["flag_tem_acordo_valor_deeplegal"] = (df_fin["valor_acordo_deeplegal"].fillna(0) > 0).astype(int)

severity_summary = pd.DataFrame([{
    "qtd_com_condenacao": int(df_fin["flag_tem_condenacao_deeplegal"].sum()),
    "valor_total_condenacao": float(df_fin["valor_condenacao_deeplegal"].sum()),
    "condenacao_media": float(df_fin["valor_condenacao_deeplegal"].mean()),
    "condenacao_mediana": float(df_fin["valor_condenacao_deeplegal"].median()),
    "qtd_com_acordo_valor": int(df_fin["flag_tem_acordo_valor_deeplegal"].sum()),
    "valor_total_acordo": float(df_fin["valor_acordo_deeplegal"].sum()),
    "acordo_medio": float(df_fin["valor_acordo_deeplegal"].mean()),
    "acordo_mediano": float(df_fin["valor_acordo_deeplegal"].median()),
    "ratio_medio_condenacao_sobre_benner": float(df_fin["ratio_condenacao_sobre_valor_benner"].replace([np.inf, -np.inf], np.nan).mean()),
    "ratio_medio_acordo_sobre_benner": float(df_fin["ratio_acordo_sobre_valor_benner"].replace([np.inf, -np.inf], np.nan).mean()),
}])

save_report(severity_summary, "severity_condenacao_acordo_deeplegal_summary.csv")
severity_summary

## 10. Métricas financeiras proxy

In [ ]:
df_fin["exposicao_benner_proxy"] = df_fin["valor_benner"].fillna(0) * df_fin["peso_estimativa_perda"].fillna(0.30)
df_fin["exposicao_deeplegal_proxy"] = df_fin["valor_deeplegal"].fillna(0) * df_fin["peso_estimativa_perda"].fillna(0.30)
df_fin["valor_max_benner_deeplegal"] = df_fin[["valor_benner", "valor_deeplegal"]].max(axis=1)
df_fin["exposicao_max_valor_proxy"] = df_fin["valor_max_benner_deeplegal"].fillna(0) * df_fin["peso_estimativa_perda"].fillna(0.30)

df_fin["perda_observada_deeplegal_proxy"] = df_fin[["valor_condenacao_deeplegal", "valor_acordo_deeplegal"]].max(axis=1).fillna(0)

df_fin["exposicao_hibrida_proxy"] = np.where(
    df_fin["perda_observada_deeplegal_proxy"] > 0,
    df_fin["perda_observada_deeplegal_proxy"],
    df_fin["exposicao_max_valor_proxy"]
)

exposure_summary = pd.DataFrame([{
    "exposicao_benner_proxy_total": float(df_fin["exposicao_benner_proxy"].sum()),
    "exposicao_deeplegal_proxy_total": float(df_fin["exposicao_deeplegal_proxy"].sum()),
    "exposicao_max_valor_proxy_total": float(df_fin["exposicao_max_valor_proxy"].sum()),
    "perda_observada_deeplegal_proxy_total": float(df_fin["perda_observada_deeplegal_proxy"].sum()),
    "exposicao_hibrida_proxy_total": float(df_fin["exposicao_hibrida_proxy"].sum()),
    "exposicao_hibrida_media": float(df_fin["exposicao_hibrida_proxy"].mean()),
    "exposicao_hibrida_mediana": float(df_fin["exposicao_hibrida_proxy"].median()),
    "exposicao_hibrida_p95": float(df_fin["exposicao_hibrida_proxy"].quantile(0.95)),
    "exposicao_hibrida_p99": float(df_fin["exposicao_hibrida_proxy"].quantile(0.99)),
}])

save_report(exposure_summary, "exposure_proxy_summary_integrated.csv")
exposure_summary

## 11. Análise financeira por grupo integrado

In [ ]:
def financial_integrated_group_report(df, group_col, min_count=30):
    if group_col not in df.columns:
        return None

    agg = {
        "qtd_processos": (df.columns[0], "count"),
        "valor_total_benner": ("valor_benner", "sum"),
        "valor_medio_benner": ("valor_benner", "mean"),
        "valor_mediano_benner": ("valor_benner", "median"),
        "valor_p90_benner": ("valor_benner", lambda x: x.quantile(0.90)),
        "valor_total_deeplegal": ("valor_deeplegal", "sum"),
        "valor_medio_deeplegal": ("valor_deeplegal", "mean"),
        "condenacao_total_deeplegal": ("valor_condenacao_deeplegal", "sum"),
        "acordo_total_deeplegal": ("valor_acordo_deeplegal", "sum"),
        "perda_observada_deeplegal_total": ("perda_observada_deeplegal_proxy", "sum"),
        "exposicao_benner_proxy_total": ("exposicao_benner_proxy", "sum"),
        "exposicao_max_valor_proxy_total": ("exposicao_max_valor_proxy", "sum"),
        "exposicao_hibrida_proxy_total": ("exposicao_hibrida_proxy", "sum"),
        "taxa_perda_possivel_ou_provavel_benner": ("flag_perda_possivel_ou_provavel", "mean"),
        "taxa_perda_provavel_benner": ("flag_perda_provavel", "mean"),
        "taxa_perda_textual_deeplegal": ("dl_target_perda_textual", "mean"),
        "taxa_match_deeplegal_processos": ("flag_match_deeplegal_processos", "mean"),
        "taxa_match_deeplegal_eventos": ("flag_match_deeplegal_eventos", "mean"),
        "qtd_divergencia_material_valor": ("flag_divergencia_material_valor", "sum"),
        "taxa_divergencia_material_valor": ("flag_divergencia_material_valor", "mean"),
        "taxa_passivel_acordo_benner": ("flag_passivel_acordo_benner", "mean"),
        "taxa_processo_relevante_benner": ("flag_processo_relevante_benner", "mean"),
    }

    for col in EVENT_SIGNAL_COLS:
        if col in df.columns:
            agg[f"taxa_{col}"] = (col, "mean")

    out = df.groupby(group_col, dropna=False).agg(**agg).reset_index()
    out = out[out["qtd_processos"] >= min_count].copy()

    total_exposure = out["exposicao_hibrida_proxy_total"].sum()
    total_value = out["valor_total_benner"].sum()

    out["share_exposicao_hibrida_proxy"] = out["exposicao_hibrida_proxy_total"] / total_exposure if total_exposure else np.nan
    out["share_valor_total_benner"] = out["valor_total_benner"] / total_value if total_value else np.nan
    out["exposicao_hibrida_media_por_processo"] = out["exposicao_hibrida_proxy_total"] / out["qtd_processos"]

    return out.sort_values("exposicao_hibrida_proxy_total", ascending=False)

group_cols = existing_cols(df_fin, [
    BENNER_CARTEIRA_COL,
    BENNER_PRODUTO_COL,
    BENNER_ACAO_COL,
    BENNER_ASSUNTO_COL,
    BENNER_TIPO_PROCESSO_COL,
    BENNER_UF_COL,
    BENNER_COMARCA_COL,
    BENNER_FASE_COL,
    BENNER_CREDENCIADO_COL,
    BENNER_ADV_INTERNO_COL,
    BENNER_LOCAL_SERVICO_COL,
    BENNER_PASSIVEL_ACORDO_COL,
    BENNER_RELEVANTE_COL,
    DL_ASSUNTO_COL,
    DL_CLASSE_COL,
    DL_UF_COL,
    DL_CIDADE_COL,
    DL_VARA_COL,
    DL_ORGAO_COL,
    DL_FASE_COL,
    DL_STATUS_COL,
    "dl_ultima_categoria_evento",
    "faixa_valor_benner",
])

financial_group_reports = {}

for col in group_cols:
    report = financial_integrated_group_report(df_fin, col, min_count=30)
    if report is not None:
        financial_group_reports[col] = report
        safe = re.sub(r"[^a-zA-Z0-9_]", "_", col)
        save_report(report, f"financial_integrated_by_{safe}.csv")
        print("\\n" + "=" * 120)
        print(col)
        display(report.head(20))

## 12. Priorização integrada por assunto Benner

In [ ]:
if BENNER_ASSUNTO_COL not in financial_group_reports:
    raise KeyError(f"Relatório por {BENNER_ASSUNTO_COL} não foi gerado.")

priority_assunto = financial_group_reports[BENNER_ASSUNTO_COL].copy()

volume_median = priority_assunto["qtd_processos"].median()
exposure_median = priority_assunto["exposicao_hibrida_proxy_total"].median()
loss_rate_median = priority_assunto["taxa_perda_possivel_ou_provavel_benner"].median()

priority_assunto["volume_alto"] = priority_assunto["qtd_processos"] >= volume_median
priority_assunto["exposicao_alta"] = priority_assunto["exposicao_hibrida_proxy_total"] >= exposure_median
priority_assunto["taxa_perda_alta"] = priority_assunto["taxa_perda_possivel_ou_provavel_benner"] >= loss_rate_median

def classify_financial_priority(row):
    if row["exposicao_alta"] and row["volume_alto"] and row["taxa_perda_alta"]:
        return "prioridade_1_atacar_agora"
    if row["exposicao_alta"] and row["volume_alto"]:
        return "prioridade_2_alta_exposicao_em_massa"
    if row["exposicao_alta"] and not row["volume_alto"]:
        return "prioridade_3_casos_alto_valor"
    if row["volume_alto"] and row["taxa_perda_alta"]:
        return "prioridade_4_melhoria_tese_ou_operacao"
    return "monitorar"

def recommend_action(row):
    p = row["prioridade_financeira_integrada"]
    acordo = row.get("taxa_passivel_acordo_benner", 0)
    execucao = row.get("taxa_dl_flag_teve_execucao", 0)
    condenacao = row.get("condenacao_total_deeplegal", 0)

    if p == "prioridade_1_atacar_agora":
        if acordo >= 0.5:
            return "Atacar agora com régua de acordo seletivo, revisão de tese e causa raiz operacional"
        return "Atacar agora com revisão de tese, provas, documentação e estratégia de defesa"
    if p == "prioridade_2_alta_exposicao_em_massa":
        return "Criar esteira de massa: automação, triagem, defesa padronizada e negociação em lote"
    if p == "prioridade_3_casos_alto_valor":
        return "Criar governança executiva e acompanhamento individual dos casos de maior valor"
    if p == "prioridade_4_melhoria_tese_ou_operacao":
        return "Investigar tese jurídica, qualidade da documentação e origem operacional da recorrência"
    if execucao >= 0.5 or condenacao > 0:
        return "Monitorar com foco em execução/condenação e avaliar recuperação ou acordo"
    return "Monitorar e reavaliar no próximo ciclo"

priority_assunto["prioridade_financeira_integrada"] = priority_assunto.apply(classify_financial_priority, axis=1)
priority_assunto["acao_recomendada"] = priority_assunto.apply(recommend_action, axis=1)

priority_assunto = priority_assunto.sort_values("exposicao_hibrida_proxy_total", ascending=False)

save_report(priority_assunto, "financial_priority_integrated_by_benner_assunto.csv")
priority_assunto.head(50)

## 13. Priorização por ação + assunto

In [ ]:
if BENNER_ACAO_COL in df_fin.columns and BENNER_ASSUNTO_COL in df_fin.columns:
    df_fin["benner_acao_assunto"] = (
        df_fin[BENNER_ACAO_COL].astype(str).fillna("NAO_INFORMADO") +
        " | " +
        df_fin[BENNER_ASSUNTO_COL].astype(str).fillna("NAO_INFORMADO")
    )

    report_acao_assunto = financial_integrated_group_report(df_fin, "benner_acao_assunto", min_count=20)
    save_report(report_acao_assunto, "financial_integrated_by_benner_acao_assunto.csv")
    display(report_acao_assunto.head(50))
else:
    print("Colunas de ação/assunto Benner não encontradas.")

## 14. Potencial de acordo

In [ ]:
df_fin["flag_acordo_deeplegal_evento"] = df_fin["dl_flag_teve_acordo"] if "dl_flag_teve_acordo" in df_fin.columns else 0
df_fin["flag_acordo_deeplegal_valor"] = (df_fin["valor_acordo_deeplegal"].fillna(0) > 0).astype(int)

df_fin["flag_oportunidade_acordo"] = (
    (df_fin["flag_passivel_acordo_benner"] == 1) &
    (df_fin["flag_acordo_deeplegal_evento"].fillna(0) == 0) &
    (df_fin["flag_acordo_deeplegal_valor"] == 0) &
    (df_fin["exposicao_hibrida_proxy"] > df_fin["exposicao_hibrida_proxy"].quantile(0.75))
).astype(int)

settlement_opportunity_summary = pd.DataFrame([{
    "qtd_passivel_acordo_benner": int(df_fin["flag_passivel_acordo_benner"].sum()),
    "valor_total_passivel_acordo_benner": float(df_fin.loc[df_fin["flag_passivel_acordo_benner"] == 1, "valor_benner"].sum()),
    "exposicao_hibrida_passivel_acordo": float(df_fin.loc[df_fin["flag_passivel_acordo_benner"] == 1, "exposicao_hibrida_proxy"].sum()),
    "qtd_com_acordo_evento_deeplegal": int(df_fin["flag_acordo_deeplegal_evento"].fillna(0).sum()),
    "qtd_com_acordo_valor_deeplegal": int(df_fin["flag_acordo_deeplegal_valor"].sum()),
    "qtd_oportunidade_acordo": int(df_fin["flag_oportunidade_acordo"].sum()),
    "exposicao_hibrida_oportunidade_acordo": float(df_fin.loc[df_fin["flag_oportunidade_acordo"] == 1, "exposicao_hibrida_proxy"].sum()),
}])

save_report(settlement_opportunity_summary, "settlement_opportunity_summary_integrated.csv")
settlement_opportunity_summary

In [ ]:
settlement_by_subject = financial_integrated_group_report(
    df_fin[df_fin["flag_passivel_acordo_benner"] == 1].copy(),
    BENNER_ASSUNTO_COL,
    min_count=10
)

if settlement_by_subject is not None:
    save_report(settlement_by_subject, "settlement_opportunity_by_benner_assunto.csv")
    display(settlement_by_subject.head(50))
else:
    print("Não foi possível gerar relatório de acordo por assunto.")

## 15. Causas de massa vs causas estratégicas

In [ ]:
valor_p75 = df_fin["valor_benner"].quantile(0.75)
valor_p90 = df_fin["valor_benner"].quantile(0.90)
exposicao_p90 = df_fin["exposicao_hibrida_proxy"].quantile(0.90)

df_fin["segmento_financeiro_processo"] = "massa_baixo_ou_medio_valor"
df_fin.loc[df_fin["valor_benner"] >= valor_p90, "segmento_financeiro_processo"] = "estrategico_alto_valor"
df_fin.loc[df_fin["exposicao_hibrida_proxy"] >= exposicao_p90, "segmento_financeiro_processo"] = "alto_risco_financeiro"
df_fin.loc[(df_fin["valor_benner"] < valor_p75) & (df_fin["flag_passivel_acordo_benner"] == 1), "segmento_financeiro_processo"] = "massa_potencial_acordo"

segment_report = (
    df_fin.groupby("segmento_financeiro_processo")
    .agg(
        qtd_processos=(df_fin.columns[0], "count"),
        valor_total_benner=("valor_benner", "sum"),
        valor_medio_benner=("valor_benner", "mean"),
        exposicao_hibrida_total=("exposicao_hibrida_proxy", "sum"),
        taxa_perda_possivel_ou_provavel=("flag_perda_possivel_ou_provavel", "mean"),
        taxa_passivel_acordo=("flag_passivel_acordo_benner", "mean"),
        taxa_match_deeplegal_processos=("flag_match_deeplegal_processos", "mean"),
        taxa_match_deeplegal_eventos=("flag_match_deeplegal_eventos", "mean"),
    )
    .reset_index()
)

segment_report["share_exposicao_hibrida"] = segment_report["exposicao_hibrida_total"] / segment_report["exposicao_hibrida_total"].sum()

save_report(segment_report, "financial_process_segment_report_integrated.csv")
segment_report

## 16. Ranking financeiro integrado dos processos

In [ ]:
df_fin["rank_exposicao_hibrida"] = df_fin["exposicao_hibrida_proxy"].rank(pct=True)
df_fin["rank_valor_benner"] = df_fin["valor_benner"].rank(pct=True)
df_fin["rank_perda_observada_deeplegal"] = df_fin["perda_observada_deeplegal_proxy"].rank(pct=True)

df_fin["score_financeiro_integrado"] = 0
df_fin["score_financeiro_integrado"] += 0.35 * df_fin["rank_exposicao_hibrida"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.20 * df_fin["rank_valor_benner"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.10 * df_fin["rank_perda_observada_deeplegal"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.10 * df_fin["flag_perda_possivel_ou_provavel"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.08 * df_fin["flag_passivel_acordo_benner"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.07 * df_fin["flag_processo_relevante_benner"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.05 * df_fin["flag_divergencia_material_valor"].fillna(0)
df_fin["score_financeiro_integrado"] += 0.05 * df_fin["flag_match_deeplegal_eventos"].fillna(0)

df_fin["score_financeiro_integrado"] = df_fin["score_financeiro_integrado"].clip(0, 1)

df_fin["prioridade_financeira_processo"] = pd.cut(
    df_fin["score_financeiro_integrado"],
    bins=[-0.01, 0.40, 0.70, 1.01],
    labels=["Baixa", "Media", "Alta"]
).astype(str)

ranking_cols = existing_cols(df_fin, [
    BENNER_ID_COL,
    BENNER_NUM_PROCESSO_COL,
    BENNER_CARTEIRA_COL,
    BENNER_PRODUTO_COL,
    BENNER_ACAO_COL,
    BENNER_ASSUNTO_COL,
    BENNER_TIPO_PROCESSO_COL,
    BENNER_UF_COL,
    BENNER_COMARCA_COL,
    BENNER_FASE_COL,
    BENNER_ESTIMATIVA_PERDA_COL,
    BENNER_PASSIVEL_ACORDO_COL,
    BENNER_RELEVANTE_COL,
    "valor_benner",
    "valor_deeplegal",
    "valor_condenacao_deeplegal",
    "valor_acordo_deeplegal",
    "perda_observada_deeplegal_proxy",
    "exposicao_hibrida_proxy",
    "flag_divergencia_material_valor",
    "segmento_financeiro_processo",
    "score_financeiro_integrado",
    "prioridade_financeira_processo",
    DL_ASSUNTO_COL,
    DL_CLASSE_COL,
    DL_FASE_COL,
    DL_STATUS_COL,
    "dl_ultima_categoria_evento",
    "dl_qtd_eventos",
])

ranking_financeiro_integrado = (
    df_fin[ranking_cols]
    .sort_values("score_financeiro_integrado", ascending=False)
)

save_report(ranking_financeiro_integrado, "ranking_financeiro_integrado_processos.csv")
ranking_financeiro_integrado.head(100)

## 17. Resumo executivo financeiro integrado

In [ ]:
executive_summary = {
    "qtd_processos": len(df_fin),
    "qtd_com_valor_benner": int(df_fin["valor_benner"].notna().sum()),
    "qtd_com_valor_deeplegal": int(df_fin["valor_deeplegal"].notna().sum()),
    "qtd_com_condenacao_deeplegal": int(df_fin["flag_tem_condenacao_deeplegal"].sum()),
    "qtd_com_acordo_valor_deeplegal": int(df_fin["flag_tem_acordo_valor_deeplegal"].sum()),
    "valor_total_benner": float(df_fin["valor_benner"].sum()),
    "valor_total_deeplegal": float(df_fin["valor_deeplegal"].sum()),
    "valor_total_condenacao_deeplegal": float(df_fin["valor_condenacao_deeplegal"].sum()),
    "valor_total_acordo_deeplegal": float(df_fin["valor_acordo_deeplegal"].sum()),
    "exposicao_benner_proxy_total": float(df_fin["exposicao_benner_proxy"].sum()),
    "exposicao_max_valor_proxy_total": float(df_fin["exposicao_max_valor_proxy"].sum()),
    "exposicao_hibrida_proxy_total": float(df_fin["exposicao_hibrida_proxy"].sum()),
    "qtd_prioridade_alta": int((df_fin["prioridade_financeira_processo"] == "Alta").sum()),
    "qtd_prioridade_media": int((df_fin["prioridade_financeira_processo"] == "Media").sum()),
    "qtd_prioridade_baixa": int((df_fin["prioridade_financeira_processo"] == "Baixa").sum()),
    "qtd_oportunidade_acordo": int(df_fin["flag_oportunidade_acordo"].sum()),
    "exposicao_oportunidade_acordo": float(df_fin.loc[df_fin["flag_oportunidade_acordo"] == 1, "exposicao_hibrida_proxy"].sum()),
    "qtd_divergencia_material_valor": int(df_fin["flag_divergencia_material_valor"].sum()),
    "perc_divergencia_material_valor": float(df_fin["flag_divergencia_material_valor"].mean()),
}

if "flag_perda_possivel_ou_provavel" in df_fin.columns:
    executive_summary["taxa_perda_possivel_ou_provavel_benner"] = float(df_fin["flag_perda_possivel_ou_provavel"].mean())

if "dl_target_perda_textual" in df_fin.columns:
    executive_summary["taxa_perda_textual_deeplegal_com_target"] = float(df_fin["dl_target_perda_textual"].mean())

executive_summary_df = pd.DataFrame([executive_summary])
save_report(executive_summary_df, "executive_summary_financeiro_integrado_benner_deeplegal.csv")
executive_summary_df

## 18. Salvar base financeira integrada

In [ ]:
df_fin.to_parquet(PROCESSED_DIR / "abt_financeira_integrada_benner_deeplegal.parquet", index=False)
ranking_financeiro_integrado.to_parquet(PROCESSED_DIR / "ranking_financeiro_integrado_processos.parquet", index=False)
priority_assunto.to_parquet(PROCESSED_DIR / "financial_priority_integrated_by_benner_assunto.parquet", index=False)

print("Base financeira integrada:", PROCESSED_DIR / "abt_financeira_integrada_benner_deeplegal.parquet")
print("Ranking financeiro:", PROCESSED_DIR / "ranking_financeiro_integrado_processos.parquet")
print("Prioridade por assunto:", PROCESSED_DIR / "financial_priority_integrated_by_benner_assunto.parquet")
print(df_fin.shape)

## 19. Validar saídas esperadas

In [ ]:
expected_outputs = [
    REPORTS_DIR / "financial_profile_integrado_benner_deeplegal.csv",
    REPORTS_DIR / "pareto_financeiro_integrado_benner_deeplegal.csv",
    REPORTS_DIR / "risk_by_faixa_valor_benner.csv",
    REPORTS_DIR / "risk_by_faixa_valor_deeplegal.csv",
    REPORTS_DIR / "value_divergence_summary_benner_deeplegal.csv",
    REPORTS_DIR / "top_300_divergencias_valor_benner_deeplegal.csv",
    REPORTS_DIR / "severity_condenacao_acordo_deeplegal_summary.csv",
    REPORTS_DIR / "exposure_proxy_summary_integrated.csv",
    REPORTS_DIR / "financial_priority_integrated_by_benner_assunto.csv",
    REPORTS_DIR / "financial_integrated_by_benner_acao_assunto.csv",
    REPORTS_DIR / "settlement_opportunity_summary_integrated.csv",
    REPORTS_DIR / "settlement_opportunity_by_benner_assunto.csv",
    REPORTS_DIR / "financial_process_segment_report_integrated.csv",
    REPORTS_DIR / "ranking_financeiro_integrado_processos.csv",
    REPORTS_DIR / "executive_summary_financeiro_integrado_benner_deeplegal.csv",
    PROCESSED_DIR / "abt_financeira_integrada_benner_deeplegal.parquet",
    PROCESSED_DIR / "ranking_financeiro_integrado_processos.parquet",
    PROCESSED_DIR / "financial_priority_integrated_by_benner_assunto.parquet",
]

for path in expected_outputs:
    print(path, "OK" if path.exists() else "NÃO ENCONTRADO")

# Roteiro de explicação da análise financeira integrada

## 1. Por que fazer uma EDA financeira integrada?

A base Benner mostra a exposição financeira interna do banco, principalmente via `Valorajuizado`, estimativa de perda, carteira, produto, ação, assunto e responsáveis.

A DeepLegal complementa com sinais externos do processo: valor da causa, condenação, acordo, sentença, recurso, execução, pagamento, trânsito em julgado, órgão julgador e eventos.

A análise integrada permite priorizar não apenas onde existe maior volume de processos, mas onde existe maior **risco econômico real ou potencial**.

## 2. Valor Benner vs Valor DeepLegal

A comparação entre `valor_benner` e `valor_deeplegal` mostra consistência ou divergência entre visão interna e externa.

Divergências relevantes podem indicar:

- erro de cadastro;
- diferença conceitual entre valor interno e valor judicial;
- defasagem de atualização;
- problema de match;
- múltiplos contratos/processos ligados a um mesmo caso.

## 3. Concentração financeira

O Pareto mostra se poucos processos concentram a maior parte da exposição. Se isso ocorrer, o banco deve tratar separadamente:

- causas de massa;
- causas estratégicas de alto valor.

## 4. Condenação e acordo

Quando há valor de condenação ou acordo na DeepLegal, esses campos funcionam como sinais de severidade observada.

Eles ajudam a calibrar:

- se o valor ajuizado é uma boa proxy de perda;
- se acordos estão sendo feitos em patamar relevante;
- quais assuntos têm maior materialização financeira.

## 5. Exposição híbrida proxy

Criamos uma métrica que usa:

- perda observada DeepLegal, quando existe;
- senão, maior valor entre Benner e DeepLegal ponderado pela estimativa de perda.

Essa métrica não é provisão oficial. É um indicador exploratório para priorização.

## 6. Priorização por assunto

O relatório por `Nm_Assunto` identifica os temas que combinam:

- alto volume;
- alta exposição;
- alta taxa de perda possível/provável;
- eventos jurídicos relevantes;
- oportunidade de acordo.

## 7. Potencial de acordo

A análise de acordo separa processos com alta exposição e marcados como passíveis de acordo, mas sem acordo identificado na DeepLegal.

Isso pode virar uma frente de atuação operacional.

## 8. Ranking de processos

O ranking final prioriza processos individuais combinando exposição, estimativa de perda, perda observada, acordo, relevância, eventos e divergência de valor.

## 9. Como usar o resultado

Os principais entregáveis para o jurídico são:

1. Top assuntos para atacar.
2. Top ações + assuntos para atacar.
3. Top processos de maior exposição.
4. Oportunidades de acordo.
5. Divergências de valor para saneamento cadastral.
6. Carteiras/produtos/fases com maior exposição econômica.